In [1]:
 # Please don't edit the lines below!
from lib.widgets import *

ultralytics = UltralyticsManager(verbose=True)
display(ultralytics)

# Not rendering correctly? Run this cell again!

In [2]:
import cv2
import PIL

# Wait for Ultralytics server to be ready and load selected models.
# NOTE: This may take a while and is totally normal!
client = ultralytics.create_client(
    models=[
        "yolov8n.pt",
        # Add more local .pt models from ./models here if needed.
    ]
)

img = PIL.Image.open("assets/fire.jpeg")
results = client.infer(img)

for result in results:
    print(result)

{'class': 'oven', 'class_id': 69, 'confidence': 0.7332705855369568, 'bbox': {'x1': 100.25861358642578, 'y1': 90.06866455078125, 'x2': 196.3804931640625, 'y2': 168.03829956054688}, 'x': 148.31955337524414, 'y': 129.05348205566406, 'width': 96.12187957763672, 'height': 77.96963500976562}


In [3]:
import time

# Camera and model settings for live inference.
CAMERA_INDEX = 0
CONFIDENCE = 0.4
MODEL = "yolov8n.pt"
WINDOW_NAME = "YOLO Inference (Notebook Client)"

# Distinct colors used to draw class bounding boxes.
COLORS = [
    (255, 56, 56),
    (255, 157, 151),
    (255, 112, 31),
    (255, 178, 29),
    (207, 210, 49),
    (72, 249, 10),
    (146, 204, 23),
    (61, 219, 134),
    (26, 147, 52),
    (0, 212, 187),
    (44, 153, 168),
    (0, 194, 255),
    (52, 69, 147),
    (100, 115, 255),
    (0, 24, 236),
    (132, 56, 255),
    (82, 0, 133),
    (203, 56, 255),
    (255, 149, 200),
    (255, 55, 199),
]

# Draw model predictions (boxes + labels) on a frame.
def draw_predictions(frame, predictions):
    for pred in predictions:
        x1 = int(pred["bbox"]["x1"])
        y1 = int(pred["bbox"]["y1"])
        x2 = int(pred["bbox"]["x2"])
        y2 = int(pred["bbox"]["y2"])
        cls_id = int(pred["class_id"])
        class_name = pred["class"]
        label = f"{class_name} {pred['confidence']:.2f}"
        color = COLORS[cls_id % len(COLORS)]

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(
            frame,
            label,
            (x1 + 2, y1 - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            1,
        )
    return frame


# Apply confidence preference for this notebook run.
client.confidence = CONFIDENCE

# Ensure stale windows from interrupted runs are closed before opening the camera again.
cv2.destroyAllWindows()
cv2.waitKey(1)

cap = None
try:
    # Open the webcam stream.
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open camera {CAMERA_INDEX}")

    print(f"Streaming camera {CAMERA_INDEX} with model '{MODEL}' (q to quit)")

    # Smoothed FPS display value.
    fps_display = 0.0
    while True:
        # Grab one frame from the webcam.
        ret, frame = cap.read()
        if not ret:
            break

        # Track full loop time for FPS.
        t0 = time.perf_counter()

        try:
            # Run inference through the high-level notebook client API.
            infer_t0 = time.perf_counter()
            predictions = client.infer(frame, model=MODEL)
            infer_ms = (time.perf_counter() - infer_t0) * 1000.0
            frame = draw_predictions(frame, predictions)
        except Exception as e:
            print(e)
            cv2.putText(
                frame,
                f"Server error: {e}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2,
            )
            infer_ms = 0.0

        # Update a smoothed FPS estimate.
        elapsed = max(time.perf_counter() - t0, 1e-6)
        fps_display = 0.9 * fps_display + 0.1 * (1.0 / elapsed)

        # Show performance metrics in the frame footer.
        cv2.putText(
            frame,
            f"FPS: {fps_display:.1f} infer: {infer_ms:.0f}ms",
            (10, frame.shape[0] - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )

        # Render the annotated frame and allow quitting with q.
        cv2.imshow(WINDOW_NAME, frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
except KeyboardInterrupt:
    print("Interrupted. Releasing camera...")
finally:
    # Release resources even if the cell is interrupted.
    if cap is not None:
        cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Streaming camera 0 with model 'yolov8n.pt' (q to quit)
